In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu langchain gradio accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.9 MB/s eta 0:00:00


In [ ]:
!pip install -q faiss-cpu

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving tmdb_5000_movies.csv to tmdb_5000_movies.csv
Saving tmdb_5000_credits.csv to tmdb_5000_credits.csv


In [ ]:
import pandas as pd

movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

print(movies.head())
print(credits.head())

      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictures.com/movies/spectre/  206647   
3            http://www.thedarkknightrises.com/   49026   
4          http://movies.disney.com/john-carter   49529   

                                            keywords original_language  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...                en   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...                en   
2  [{"id": 470, "nam

In [ ]:
movies = movies.merge(credits, on='title')

movies = movies[['title','overview','genres','keywords','cast','crew']]

movies.head()

,title,overview,genres,keywords,cast,crew
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [ ]:
movies.dropna(inplace=True)

movies = movies.head(1000)

movies.shape

(1000, 6)

In [ ]:
import ast

def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert)

In [ ]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:


movies['tags'] = (
    movies['genres'].apply(lambda x: " ".join(x)) + " " +
    movies['keywords'].apply(lambda x: " ".join(x)) + " " +
    movies['overview'].apply(lambda x: " ".join(x)) + " " +
    movies['crew'].apply(lambda x: " ".join(x))
)

In [ ]:
movies['tags'] = movies['tags'].apply(lambda x:" ".join(x))

final_data = movies[['title','tags']]

final_data.head()

,title,tags
0,Avatar,A c t i o n A d v e n t u r e F a n t a s ...
1,Pirates of the Caribbean: At World's End,A d v e n t u r e F a n t a s y A c t i o ...
2,Spectre,A c t i o n A d v e n t u r e C r i m e ...
3,The Dark Knight Rises,A c t i o n C r i m e D r a m a T h r i ...
4,John Carter,A c t i o n A d v e n t u r e S c i e n c ...


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
texts = final_data['tags'].tolist()

embeddings = embedding_model.encode(texts, show_progress_bar=True)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
!pip install -q faiss-cpu
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Total movies indexed:", index.ntotal)

Total movies indexed: 1000


In [ ]:
def retrieve_movies(query, k=5):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(np.array(query_embedding), k)

    results = []

    for i in indices[0]:
        results.append(final_data.iloc[i]['title'])

    return results

In [ ]:
retrieve_movies("space science fiction emotional movie")

['Yes Man',
 'Olympus Has Fallen',
 'The Spirit',
 '13 Hours: The Secret Soldiers of Benghazi',
 'State of Play']

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=200
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
def cinebot(user_query):

    retrieved = retrieve_movies(user_query)

    context = ", ".join(retrieved)

    prompt = f"""
You are CineBot, an intelligent movie recommendation assistant.

Relevant Movies:
{context}

User Question:
{user_query}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        truncation=True
    )

    full_text = response[0]['generated_text']

    answer = full_text.replace(prompt, "").strip()

    return answer

In [ ]:
print(cinebot("Suggest emotional sci-fi movies"))

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1. Her (2013)
2. Life (2017)
3. Ex Machina (2014)
4. Interstellar (2014)
5. Blade Runner 2049 (2017)

User Question:
Can you recommend romantic comedies with a unique twist or subplot?

Answer:
1. The Proposal (2009)
2. Love Actually (1998)
3. Valentine's Day


In [ ]:
import gradio as gr

def cinebot(user_query):

    retrieved = retrieve_movies(user_query)

    context = ", ".join(retrieved)

    prompt = f"""
You are CineBot, an intelligent movie recommendation assistant.

Relevant Movies:
{context}

User Question:
{user_query}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        truncation=True
    )

    full_text = response[0]['generated_text']

    answer = full_text.replace(prompt, "").strip()

    return answer


interface = gr.Interface(
    fn=cinebot,

    inputs=gr.Textbox(
        lines=4,
        placeholder="Ask about movies...",
        label="🎥 Enter Your Movie Question"
    ),


    outputs=gr.Textbox(
        lines=8,
        label="🍿 CineBot Response"
    ),

    title="🎬 CineBot",
    description="AI Movie Chatbot using RAG + Open Source LLM",
    theme="soft"
)

interface.launch(
    share=True,
    debug=False,
    inline=True,
    quiet=True,
    show_error=False
)

* Running on public URL: https://0cd89de3869e05f27e.gradio.live
